## Week 5 — Self-Improving RAG (mmaitsimwale)

### What week 5 covers
- **Day 1:** keyword dict-lookup RAG; `gpt-4.1-nano`, no embeddings.
- **Day 2:** `RecursiveCharacterTextSplitter` → `HuggingFaceEmbeddings(all-MiniLM-L6-v2)` → Chroma `vector_db`; t-SNE visualisation.
- **Day 3:** LangChain `as_retriever()` + `ChatOpenAI`; basic system-prompt RAG.
- **Day 4:** Evaluation harness — 150 test cases, MRR / nDCG / keyword coverage for retrieval, LLM-as-judge (Accuracy / Completeness / Relevance).
- **Day 5:** No-LangChain advanced RAG: LLM chunking (`headline + summary + original_text`) → `text-embedding-3-large` → Chroma `PersistentClient` → LLM reranking → query rewriting.

### What this exercise adds — the Self-Improvement Loop
For each query:
1. Rewrite query → retrieve → rerank → generate answer.
2. Evaluate with LLM-as-judge (Accuracy / Completeness / Relevance).
3. If any score < 4.5/5: inject evaluator feedback into the next prompt and retry.
4. Repeat up to `MAX_ITERATIONS` or until **all scores ≥ 4.5/5**.


## Algorithms

1. **LLM-guided chunking (Day 5):** the LLM decomposes each markdown file into semantically coherent `Chunk(headline, summary, original_text)` objects. Each chunk is stored with headline + summary prepended, boosting retrieval signal beyond raw text.
2. **Dense retrieval:** query is embedded with `text-embedding-3-large`; Chroma cosine search returns top-k chunks.
3. **LLM reranking (Day 5 `RankOrder`):** a second LLM call re-orders retrieved chunks by relevance to the question, pushing the most useful context to the top.
4. **Query rewriting (Day 5):** the user question is rewritten into a tighter knowledge-base query before embedding.
5. **LLM-as-judge (Day 4 `AnswerEval`):** scores Accuracy / Completeness / Relevance on a 1–5 scale using the reference answer.
6. **Self-improvement loop:** carries the evaluator's own feedback verbatim into the next generation prompt, focusing regeneration on the weakest dimension.


In [ ]:
# Setup: imports, repo-root resolution, constants
import os
import sys
import json
import math
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from litellm import completion
from tqdm import tqdm


def _repo_root() -> Path:
    """Walk up from cwd until week5/knowledge-base is found (same as Week4_EXERCISE)."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "week5" / "knowledge-base").is_dir():
            return cand
    raise RuntimeError(
        "Cannot find week5/knowledge-base — start Jupyter from the llm_engineering repo root."
    )


REPO_ROOT = _repo_root()
KNOWLEDGE_BASE = REPO_ROOT / "week5" / "knowledge-base"
TESTS_FILE = REPO_ROOT / "week5" / "evaluation" / "tests.jsonl"
DB_NAME = str(REPO_ROOT / "week5" / "vector_db_exercise")   # separate from course vector_db

MODEL = "gpt-4.1-nano"
EMBEDDING_MODEL = "text-embedding-3-large"
COLLECTION_NAME = "docs_exercise"
RETRIEVAL_K = 10
MAX_ITERATIONS = 3          # max self-improvement passes per question
SUCCESS_THRESHOLD = 4.5     # minimum score (out of 5) on all dimensions

print(f"Repo root  : {REPO_ROOT}")
print(f"KB path    : {KNOWLEDGE_BASE}")
print(f"Tests file : {TESTS_FILE}")
print(f"Vector DB  : {DB_NAME}")


In [ ]:
# .env from repo root — same key pattern as week5/day5.ipynb
load_dotenv(REPO_ROOT / ".env", override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set — required for embeddings, chunking and evaluation")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (optional)")


In [ ]:
# OpenAI client (used for embeddings; litellm handles chat completions)
openai_client = OpenAI()


In [ ]:
# Pydantic models — Day 5 chunking + reranking + Day 4 evaluation


class Result(BaseModel):
    """Mirrors LangChain Document for downstream compatibility."""
    page_content: str
    metadata: dict


class Chunk(BaseModel):
    """LLM-generated semantic chunk (day5 pattern)."""
    headline: str = Field(
        description="Brief heading for this chunk (a few words), optimised for retrieval"
    )
    summary: str = Field(
        description="A few sentences summarising this chunk to answer common questions"
    )
    original_text: str = Field(
        description="The original text of this chunk, exactly as-is, unchanged"
    )

    def as_result(self, document: dict) -> Result:
        content = self.headline + "\n\n" + self.summary + "\n\n" + self.original_text
        return Result(
            page_content=content,
            metadata={"source": document["source"], "type": document["type"]},
        )


class Chunks(BaseModel):
    chunks: list[Chunk]


class RankOrder(BaseModel):
    """Ordered chunk IDs by relevance (day5 reranking structured output)."""
    order: list[int] = Field(
        description="Chunk IDs from most relevant to least relevant"
    )


class AnswerEval(BaseModel):
    """LLM-as-judge scores and feedback (day4 eval harness pattern)."""
    feedback: str = Field(
        description="Concise feedback on answer quality vs reference answer"
    )
    accuracy: float = Field(
        description="Factual correctness: 1 (wrong) to 5 (perfect)"
    )
    completeness: float = Field(
        description="Coverage of all required information: 1 (very poor) to 5 (ideal)"
    )
    relevance: float = Field(
        description="How directly the answer addresses the question: 1 (off-topic) to 5 (ideal)"
    )


class TestQuestion(BaseModel):
    """Test question with reference answer (day4 pattern)."""
    question: str
    keywords: list[str]
    reference_answer: str
    category: str


In [ ]:
# Ingest pipeline: fetch documents → LLM chunking → embed → persist to Chroma

AVERAGE_CHUNK_SIZE = 500   # hint for how many chunks the LLM should produce


def fetch_documents() -> list[dict]:
    """Load all markdown files from the knowledge base (day5 pattern)."""
    documents = []
    for folder in KNOWLEDGE_BASE.iterdir():
        if not folder.is_dir():
            continue
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append(
                    {"type": doc_type, "source": file.as_posix(), "text": f.read()}
                )
    print(f"Loaded {len(documents)} documents from {KNOWLEDGE_BASE}")
    return documents


def make_chunking_prompt(document: dict) -> str:
    """Build the LLM chunking prompt (day5 pattern with overlap hint)."""
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and split it into overlapping chunks for a Knowledge Base.

The document is from the shared drive of a company called Insurellm.
Document type : {document["type"]}
Document path : {document["source"]}

A chatbot will use these chunks to answer questions about the company.
Divide the document as you see fit; the entire document must be covered.
This document should probably be split into approximately {how_many} chunks (more or less is fine).
Include ~25% overlap (~50 words) between adjacent chunks for better retrieval.

For each chunk provide:
  - headline : a brief heading most likely to surface in a query
  - summary  : a few sentences summarising the chunk
  - original_text : the original text of the chunk, exactly as-is

Together, your chunks must represent the entire document.

Document text:
{document["text"]}

Respond with the chunks.
"""


def process_document(document: dict) -> list[Result]:
    """LLM-guided chunking for a single document (day5 pattern)."""
    messages = [{"role": "user", "content": make_chunking_prompt(document)}]
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    chunks = Chunks.model_validate_json(response.choices[0].message.content).chunks
    return [chunk.as_result(document) for chunk in chunks]


def create_chunks(documents: list[dict]) -> list[Result]:
    """Chunk all documents via the LLM."""
    all_chunks: list[Result] = []
    for doc in tqdm(documents, desc="LLM chunking"):
        try:
            all_chunks.extend(process_document(doc))
        except Exception as e:
            print(f"  Warning: could not chunk {doc['source']}: {e}")
    return all_chunks


def build_vectorstore(force_rebuild: bool = False) -> PersistentClient:
    """Build or reuse the Chroma vector store.

    Skips rebuilding if the collection already contains documents (slow LLM
    chunking + embedding step). Pass force_rebuild=True to start fresh.
    """
    chroma = PersistentClient(path=DB_NAME)
    existing = [c.name for c in chroma.list_collections()]

    if COLLECTION_NAME in existing and not force_rebuild:
        col = chroma.get_collection(COLLECTION_NAME)
        if col.count() > 0:
            print(f"Loaded existing collection '{COLLECTION_NAME}' ({col.count()} vectors). "
                  f"Pass force_rebuild=True to rebuild.")
            return chroma

    if COLLECTION_NAME in existing:
        chroma.delete_collection(COLLECTION_NAME)

    documents = fetch_documents()
    chunks = create_chunks(documents)

    texts = [c.page_content for c in chunks]
    metas = [c.metadata for c in chunks]

    # Embed in batches to respect API limits
    batch_size = 100
    all_vectors: list = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch = texts[i : i + batch_size]
        emb_data = openai_client.embeddings.create(
            model=EMBEDDING_MODEL, input=batch
        ).data
        all_vectors.extend([e.embedding for e in emb_data])

    col = chroma.get_or_create_collection(COLLECTION_NAME)
    ids = [str(i) for i in range(len(chunks))]
    col.add(ids=ids, embeddings=all_vectors, documents=texts, metadatas=metas)
    print(f"Built '{COLLECTION_NAME}' with {col.count()} vectors.")
    return chroma


chroma_client = build_vectorstore()
collection = chroma_client.get_collection(COLLECTION_NAME)


In [ ]:
# Retrieval: vector search → LLM rerank (day5 pattern)


def fetch_context_unranked(question: str, k: int = RETRIEVAL_K) -> list[Result]:
    """Embed question; return top-k chunks by cosine similarity."""
    query_vec = (
        openai_client.embeddings.create(model=EMBEDDING_MODEL, input=[question])
        .data[0]
        .embedding
    )
    results = collection.query(query_embeddings=[query_vec], n_results=k)
    return [
        Result(page_content=doc, metadata=meta)
        for doc, meta in zip(results["documents"][0], results["metadatas"][0])
    ]


def rerank(question: str, chunks: list[Result]) -> list[Result]:
    """LLM-based reranking using RankOrder structured output (day5 pattern)."""
    system_prompt = (
        "You are a document re-ranker.\n"
        "Rank the provided chunks by relevance to the question, most relevant first.\n"
        "Reply only with the ordered list of chunk IDs (integers), nothing else."
    )
    user_prompt = f"Question: {question}\n\nChunks:\n\n"
    for idx, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {idx + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the ranked chunk IDs, nothing else."

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    order = RankOrder.model_validate_json(response.choices[0].message.content).order
    # Guard against out-of-range indices from the model
    return [chunks[i - 1] for i in order if 1 <= i <= len(chunks)]


def fetch_context(question: str) -> list[Result]:
    """Full retrieval pipeline: unranked vector search → LLM rerank."""
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)


In [ ]:
# Query rewriting — tighter knowledge-base query (day5 pattern)


def rewrite_query(question: str, prior_answers: list[str] | None = None) -> str:
    """Rewrite the user question into an optimal KB search query.

    prior_answers: list of previous answers in the same session (for context).
    """
    prior_answers = prior_answers or []
    history_text = "\n".join(prior_answers[-2:]) if prior_answers else "(none)"
    message = f"""
You are answering questions about the company Insurellm.
You are about to search a Knowledge Base.

Prior answers in this conversation (for context):
{history_text}

User's current question:
{question}

Respond ONLY with a single, short, specific query string that will surface the most
relevant content in the Knowledge Base.
- Focus on the question details.
- Do not mention the company name unless it is a general company question.

IMPORTANT: Reply with ONLY the search query, nothing else.
"""
    response = completion(
        model=MODEL, messages=[{"role": "system", "content": message}]
    )
    return response.choices[0].message.content.strip()


In [ ]:
# Evaluation helpers — inline (no dependency on week5/evaluation/ module path)


def load_tests() -> list[TestQuestion]:
    """Load test questions from JSONL file."""
    tests: list[TestQuestion] = []
    with open(TESTS_FILE, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                tests.append(TestQuestion(**json.loads(line)))
    print(f"Loaded {len(tests)} test questions from {TESTS_FILE}")
    return tests


def evaluate_answer_llm(
    question: str, generated: str, reference: str
) -> AnswerEval:
    """LLM-as-judge evaluation (mirrors eval.py lines 130-158)."""
    judge_messages = [
        {
            "role": "system",
            "content": (
                "You are an expert evaluator assessing the quality of answers. "
                "Evaluate the generated answer by comparing it to the reference answer. "
                "Only give 5/5 scores for perfect answers."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Question:\n{question}\n\n"
                f"Generated Answer:\n{generated}\n\n"
                f"Reference Answer:\n{reference}\n\n"
                "Please evaluate the generated answer on three dimensions:\n"
                "1. Accuracy: How factually correct is it compared to the reference? "
                "Only give 5/5 for perfect answers. If the answer is wrong, accuracy must be 1.\n"
                "2. Completeness: How thoroughly does it address all aspects, covering all "
                "information from the reference answer?\n"
                "3. Relevance: How well does it directly answer the specific question, "
                "giving no additional information?\n\n"
                "Provide detailed feedback and scores from 1 (very poor) to 5 (ideal) "
                "for each dimension."
            ),
        },
    ]
    resp = completion(model=MODEL, messages=judge_messages, response_format=AnswerEval)
    return AnswerEval.model_validate_json(resp.choices[0].message.content)


def identify_weakness(eval_result: AnswerEval) -> str:
    """Return the name of the dimension with the lowest score."""
    scores = {
        "accuracy": eval_result.accuracy,
        "completeness": eval_result.completeness,
        "relevance": eval_result.relevance,
    }
    return min(scores, key=lambda k: scores[k])


In [ ]:
# Answer generation with iteration-aware prompts (Day 3/5 patterns)

SYSTEM_PROMPT_BASE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
Your answer will be evaluated for accuracy, relevance, and completeness.
Answer ONLY from the provided context. If the answer is not in the context, say so.
Do NOT hallucinate. Be precise and complete.

Context (extracts from the Knowledge Base):
{context}

With this context, answer the user's question accurately, completely, and relevantly.
"""

SYSTEM_PROMPT_REFINE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
Your answer will be evaluated for accuracy, relevance, and completeness.
Answer ONLY from the provided context. Do NOT hallucinate. Be precise and complete.

A previous attempt at this question received these scores:
  Accuracy     : {accuracy}/5
  Completeness : {completeness}/5
  Relevance    : {relevance}/5

Evaluator feedback: {feedback}

Your primary improvement focus for this attempt: {weakness}.
Improve specifically on {weakness}. Do NOT include information not supported by the context.

Context (extracts from the Knowledge Base):
{context}

With this context, provide an improved answer.
"""


def make_rag_messages(
    question: str,
    chunks: list[Result],
    iteration: int = 1,
    prior_eval: AnswerEval | None = None,
    weakness: str = "",
) -> list[dict]:
    """Build the messages list for the LLM call."""
    context = "\n\n".join(
        f"Extract from {c.metadata['source']}:\n{c.page_content}" for c in chunks
    )
    if iteration == 1 or prior_eval is None:
        system_content = SYSTEM_PROMPT_BASE.format(context=context)
    else:
        system_content = SYSTEM_PROMPT_REFINE.format(
            context=context,
            accuracy=prior_eval.accuracy,
            completeness=prior_eval.completeness,
            relevance=prior_eval.relevance,
            feedback=prior_eval.feedback,
            weakness=weakness,
        )
    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": question},
    ]


def generate_answer(
    question: str,
    chunks: list[Result],
    iteration: int = 1,
    prior_eval: AnswerEval | None = None,
    weakness: str = "",
) -> str:
    """Call the LLM and return the answer string."""
    messages = make_rag_messages(question, chunks, iteration, prior_eval, weakness)
    resp = completion(model=MODEL, messages=messages)
    return resp.choices[0].message.content


In [ ]:
# Self-improvement loop — the core contribution of this exercise


def _print_iteration(iteration: int, answer: str, ev: AnswerEval) -> None:
    print(f"\n{'='*64}")
    print(f"  Iteration {iteration}")
    print(f"{'='*64}")
    print(f"Answer:\n{answer}\n")
    print(
        f"Scores — Accuracy: {ev.accuracy:.1f}/5  "
        f"Completeness: {ev.completeness:.1f}/5  "
        f"Relevance: {ev.relevance:.1f}/5"
    )
    print(f"Feedback: {ev.feedback}")


def self_improving_answer(
    question: str,
    reference_answer: str,
    verbose: bool = True,
) -> dict:
    """
    Iteratively generate and improve a RAG answer.

    Process per iteration:
      1. Rewrite query using prior answers as context.
      2. Retrieve + rerank context chunks.
      3. Generate answer (iteration 1: base prompt; 2+: feedback-injected prompt).
      4. Evaluate with LLM-as-judge against the reference answer.
      5. Stop if all scores >= SUCCESS_THRESHOLD or MAX_ITERATIONS reached.
      6. Otherwise: record feedback, identify weakest dimension, retry.

    Returns a dict with: question, answer, eval (AnswerEval), iteration, converged.
    """
    prior_answers: list[str] = []
    prior_eval: AnswerEval | None = None
    weakness: str = ""
    final_result: dict = {}

    for iteration in range(1, MAX_ITERATIONS + 1):
        # Step 1: query rewriting incorporates prior answers for context
        query = rewrite_query(question, prior_answers)

        # Step 2: retrieve + rerank
        chunks = fetch_context(query)

        # Step 3: generate (plain on iteration 1, feedback-guided thereafter)
        answer = generate_answer(question, chunks, iteration, prior_eval, weakness)

        # Step 4: evaluate against reference
        eval_result = evaluate_answer_llm(question, answer, reference_answer)

        if verbose:
            _print_iteration(iteration, answer, eval_result)

        final_result = {
            "question": question,
            "answer": answer,
            "eval": eval_result,
            "iteration": iteration,
            "converged": False,
        }

        # Step 5: success condition
        scores = [eval_result.accuracy, eval_result.completeness, eval_result.relevance]
        if all(s >= SUCCESS_THRESHOLD for s in scores):
            final_result["converged"] = True
            if verbose:
                print(
                    f"\n  Converged at iteration {iteration} "
                    f"(all scores >= {SUCCESS_THRESHOLD}/5)"
                )
            break

        # Step 6: prepare for next iteration
        prior_answers.append(answer)
        prior_eval = eval_result
        weakness = identify_weakness(eval_result)
        if verbose and iteration < MAX_ITERATIONS:
            print(f"\n  -> Retrying: focusing improvement on '{weakness}'")

    return final_result


In [ ]:
# Sample run: self-improving RAG on 5 test questions
# Increase SAMPLE_SIZE to evaluate more; full suite is 150 questions.

tests = load_tests()
SAMPLE_SIZE = 5
SAMPLE = tests[:SAMPLE_SIZE]

print(f"Running self-improving RAG on {SAMPLE_SIZE} test questions "
      f"(up to {MAX_ITERATIONS} iterations each, threshold {SUCCESS_THRESHOLD}/5).\n")

all_results: list[dict] = []
for idx, t in enumerate(SAMPLE):
    print(f"\n{'#' * 64}")
    print(f"  Q{idx + 1}: {t.question}  [{t.category}]")
    print(f"{'#' * 64}")
    result = self_improving_answer(t.question, t.reference_answer)
    result["category"] = t.category
    all_results.append(result)


In [ ]:
# Results summary table

print(f"\n{'=' * 70}")
print(f"{'RESULTS SUMMARY':^70}")
print(f"{'=' * 70}")
header = (
    f"  {'Q':<2} {'Category':<18} {'Iter':>4}  "
    f"{'Acc':>5} {'Comp':>5} {'Rel':>5}  {'Converged'}"
)
print(header)
print("-" * 70)

for i, r in enumerate(all_results, 1):
    ev = r["eval"]
    converged_str = "YES" if r["converged"] else "NO"
    print(
        f"  {i:<2} {r['category']:<18} {r['iteration']:>4}  "
        f"{ev.accuracy:>5.1f} {ev.completeness:>5.1f} {ev.relevance:>5.1f}  "
        f"{converged_str}"
    )

print("-" * 70)

avg_acc  = sum(r["eval"].accuracy      for r in all_results) / len(all_results)
avg_comp = sum(r["eval"].completeness  for r in all_results) / len(all_results)
avg_rel  = sum(r["eval"].relevance     for r in all_results) / len(all_results)
n_conv   = sum(1 for r in all_results if r["converged"])

print(
    f"  {'AVERAGES':<22}      {avg_acc:>5.1f} {avg_comp:>5.1f} {avg_rel:>5.1f}"
)
print(f"\n  Converged: {n_conv}/{len(all_results)} questions reached "
      f">= {SUCCESS_THRESHOLD}/5 on all dimensions.")
print(f"\n  Key improvements applied:")
print(f"    - LLM-guided chunking (headline + summary + original text)")
print(f"    - text-embedding-3-large for dense retrieval")
print(f"    - LLM reranking (RankOrder structured output)")
print(f"    - Query rewriting before each retrieval")
print(f"    - Iterative self-improvement driven by evaluator feedback")
